# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. The dataset contains mass spectrometry records for human extracellular vesicles.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show metadata (dataset.metadata is an object, not a dict)
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their IDs, fields, and column structure.

**Entities referenced by their `@id`.**

In [ ]:
# List all record sets (@id and name)
print("Record Sets:")
record_sets = {rs['@id']: rs for rs in dataset.metadata.record_sets}
for rs_id, rs in record_sets.items():
    print(f"- @id: {rs_id} | name: {rs.get('name', 'N/A')}")

# For each record set, show its fields and columns by @id
for rs_id, rs in record_sets.items():
    print(f"\nRecord set {rs_id} ({rs.get('name', 'N/A')}):")
    fields = rs.get('fields', [])
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        # Look up the field object by id in metadata.fields
        field_obj = next((f for f in dataset.metadata.fields if f['@id'] == field_id), None)
        print(f"  Field: {field_id}")
        if field_obj is not None:
            columns = field_obj.get('columns', [])
            for column in columns:
                column_id = column['@id'] if isinstance(column, dict) else column
                print(f"    Column: {column_id}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s from above.

In [ ]:
# Collect all record set ids
record_set_ids = list(record_sets.keys())
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {record_set_id}")
    else:
        print(f"No records found for {record_set_id}")

# Show columns for the primary record set (choose the first one)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nAvailable columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No usable record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records, normalize numeric fields, group records, etc.

All fields referenced by their respective `@id`.

In [ ]:
import numpy as np

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Working with record set: {main_record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    
    # Select numeric field (@id or as matches possible names)
    # For demonstration, select the first float/int column
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        print("No numeric fields found.")
    else:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        
        threshold = df[numeric_field_id].mean() if pd.notna(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (mean): {len(filtered_df)} records")
        print(filtered_df.head())
        
        # Normalize the column
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nFirst few normalized values:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a likely categorical field
        group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"\nGrouping by categorical field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std', 'min', 'max'])
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No suitable data for EDA.")

## 5. Visualization
Visualize data distributions and relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        field_to_plot = numeric_fields[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[field_to_plot].dropna(), kde=True)
        plt.title(f"Distribution of {field_to_plot}")
        plt.xlabel(field_to_plot)
        plt.show()
        
        # If at least two numeric fields, plot their relationship
        if len(numeric_fields) > 1:
            plt.figure(figsize=(7, 7))
            sns.scatterplot(
                x=df[numeric_fields[0]], y=df[numeric_fields[1]]
            )
            plt.xlabel(numeric_fields[0])
            plt.ylabel(numeric_fields[1])
            plt.title(f"{numeric_fields[0]} vs {numeric_fields[1]}")
            plt.show()
    else:
        print("No numeric fields to visualize.")

## 6. Conclusion
Key findings from exploring the dataset:
- The FAIR^2 dataset provides mass spectrometry details of proteins from human mast cell extracellular vesicles.
- Using `mlcroissant`, data can be loaded by record set and fields referenced by their `@id`s.
- Numeric and categorical exploration are possible after loading.
- The schema allows easy traceability and reproducibility.

Further analysis (e.g., machine learning or deeper domain EDA) can be performed following these initial steps.